In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


In [2]:
epochs = 10
batch_size = 32
learning_rate = 0.001

train_transform = transforms.Compose(
    [
        transforms.Resize((64, 64)),  
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
    ]
)



In [3]:
train_dataset = ImageFolder(root=r'auto data/train', transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


In [4]:
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_set, val_set = torch.utils.data.random_split(train_dataset, [train_size, val_size])

In [5]:
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

In [6]:
print("Classes:", train_dataset.classes)
print("Total train images:", len(train_dataset))
print("Total validation images:", len(val_set))
print("Total training images:", len(train_set))


Classes: ['F', 'L', 'O', 'W']
Total train images: 12000
Total validation images: 2400
Total training images: 9600


In [7]:
images, labels = next(iter(train_loader))
print(f"Batch images shape: {images.shape}")
print(f"Batch labels shape: {labels.shape}")

Batch images shape: torch.Size([32, 3, 64, 64])
Batch labels shape: torch.Size([32])


In [8]:
class auto_movement(nn.Module):
    def __init__(self):
        super(auto_movement, self).__init__()
        self.conv1 = nn.Conv2d(3,64,3,1,1)
        self.bn1=nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(64,128,3,1,1)
        self.bn2=nn.BatchNorm2d(128)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(128*16*16,256)
        self.fc2 = nn.Linear(256,64)
        self.fc3 = nn.Linear(64,32)
        self.fc4 = nn.Linear(32,4)   
        
        
    def forward(self, x):
         x= torch.relu(self.bn1(self.conv1(x)))
         x= self.maxpool(x)
         x= torch.relu(self.bn2(self.conv2(x)))
         x= self.maxpool(x)
         x=x.view(-1, 128*16*16)
         x= torch.relu(self.fc1(x))
         x= self.dropout(x)
         x= torch.relu(self.fc2(x))
         x = self.dropout(x)
         x= torch.relu(self.fc3(x))
         x= self.dropout(x)
         x= self.fc4(x)
         return x

In [9]:
model = auto_movement()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

In [10]:
total_steps = len(train_loader)
for epoch in range(epochs):
    for i, (images,labels) in enumerate(train_loader):
        outputs = model(images)
        loss = loss_function(outputs,labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{i+1}/{total_steps}], Loss: {loss.item():.4f}')

Epoch [1/10], Step [50/300], Loss: 1.0051
Epoch [1/10], Step [100/300], Loss: 0.8450
Epoch [1/10], Step [150/300], Loss: 0.4194
Epoch [1/10], Step [200/300], Loss: 0.5077
Epoch [1/10], Step [250/300], Loss: 0.4815
Epoch [1/10], Step [300/300], Loss: 0.6445
Epoch [2/10], Step [50/300], Loss: 0.1766
Epoch [2/10], Step [100/300], Loss: 0.2492
Epoch [2/10], Step [150/300], Loss: 0.0613
Epoch [2/10], Step [200/300], Loss: 0.0951
Epoch [2/10], Step [250/300], Loss: 0.0611
Epoch [2/10], Step [300/300], Loss: 0.0066
Epoch [3/10], Step [50/300], Loss: 0.1454
Epoch [3/10], Step [100/300], Loss: 0.0581
Epoch [3/10], Step [150/300], Loss: 0.0367
Epoch [3/10], Step [200/300], Loss: 0.0108
Epoch [3/10], Step [250/300], Loss: 0.0514
Epoch [3/10], Step [300/300], Loss: 0.0198
Epoch [4/10], Step [50/300], Loss: 0.0212
Epoch [4/10], Step [100/300], Loss: 0.0319
Epoch [4/10], Step [150/300], Loss: 0.0139
Epoch [4/10], Step [200/300], Loss: 0.0891
Epoch [4/10], Step [250/300], Loss: 0.0141
Epoch [4/10], S

In [11]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = model(images)
        _,predicted= torch.max(outputs.data,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        acc= 100 * (correct/total)

In [12]:
print(f'Accuracy of the model on the test images: {acc:.2f}%')

Accuracy of the model on the test images: 99.38%


In [13]:
torch.save(model.state_dict(), 'hand_model_4classes_V2.pth')
print("Model saved successfully!")

Model saved successfully!
